# Train Flatmate RL with TRL GRPO

This notebook uses the live Hugging Face Space endpoint as the reward source for GRPO. It first collects prompt states from websocket rollouts, then trains a model to generate one JSON action for each observation. The main reward function resets the endpoint, replays the stored action prefix, applies the model action, and scores the resulting Flatmate RL transition.

Space endpoint: `https://huggingface.co/spaces/kushalExplores/flatmate_rl`

In [ ]:
# Restart the kernel after this cell if your notebook runtime asks you to.
%pip install -q "trl>=0.26.2" "transformers>=4.57.0" accelerate datasets peft websockets huggingface_hub matplotlib pandas

In [ ]:
from __future__ import annotations

import asyncio
import json
import random
import threading
from dataclasses import dataclass
from typing import Any
from urllib.parse import urlparse

import websockets
from datasets import Dataset

SPACE_HTTP_URL = "https://kushalexplores-flatmate-rl.hf.space"
SCENARIOS = [
    "task_visit_single",
    "task_visit_single_hidden_flex",
    "task_visit_multi",
    "task_visit_single_seller_followup",
]

def ws_url_from_http(base_url: str) -> str:
    parsed = urlparse(base_url.rstrip("/"))
    scheme = "wss" if parsed.scheme == "https" else "ws"
    return f"{scheme}://{parsed.netloc}/ws"

SPACE_WS_URL = ws_url_from_http(SPACE_HTTP_URL)
SPACE_WS_URL

## Websocket Environment Client

OpenEnv keeps episode state on `/ws`. The plain HTTP `/reset` and `/step` endpoints create fresh environment instances, so GRPO reward replay uses websocket sessions.

In [ ]:
class FlatmateEndpoint:
    def __init__(self, ws_url: str = SPACE_WS_URL, timeout_s: float = 120.0):
        self.ws_url = ws_url
        self.timeout_s = timeout_s

    async def __aenter__(self):
        self.ws = await websockets.connect(
            self.ws_url,
            open_timeout=self.timeout_s,
            ping_timeout=self.timeout_s,
        )
        return self

    async def __aexit__(self, exc_type, exc, tb):
        try:
            await self.ws.send(json.dumps({"type": "close"}))
        finally:
            await self.ws.close()

    async def _send(self, payload: dict[str, Any]) -> dict[str, Any]:
        await self.ws.send(json.dumps(payload))
        raw = await asyncio.wait_for(self.ws.recv(), timeout=self.timeout_s)
        message = json.loads(raw)
        if message.get("type") == "error":
            raise RuntimeError(message.get("data", message))
        data = message["data"]
        obs = data.get("observation", {})
        obs["reward"] = data.get("reward")
        obs["done"] = data.get("done", False)
        return obs

    async def reset(self, scenario_id: str, seed: int | None = None) -> dict[str, Any]:
        data: dict[str, Any] = {"scenario_id": scenario_id}
        if seed is not None:
            data["seed"] = seed
        return await self._send({"type": "reset", "data": data})

    async def step(self, action: dict[str, Any]) -> dict[str, Any]:
        return await self._send({"type": "step", "data": action})

async def smoke_test_endpoint():
    async with FlatmateEndpoint() as env:
        obs = await env.reset("task_visit_single", seed=1)
        print(obs["scenario_id"], obs["status"])
        print(obs.get("last_user_message") or obs.get("current_user_request"))

await smoke_test_endpoint()

## Prompt States

GRPO needs prompts. These prompts are endpoint observations collected from heuristic rollouts. Each row also stores the action prefix needed to reconstruct that exact state during reward scoring.

In [ ]:
def trace_tool_names(obs: dict[str, Any]) -> list[str]:
    return [str(t.get("tool", t.get("tool_name", ""))) for t in obs.get("tool_trace", [])]

def heuristic_action(obs: dict[str, Any]) -> dict[str, Any] | None:
    tools = trace_tool_names(obs)
    phase = obs.get("phase", "buyer")
    remaining = set(obs.get("remaining_required_fields", []))
    scenario_id = obs.get("scenario_id", "task_visit_single")

    if phase == "seller" and not obs.get("seller_profile_stored"):
        if remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share the household dietary setup, who the flat is for, and available visit time slots."}
        return {"action_type": "tool_call", "tool_name": "store_seller_details", "tool_arguments": {}}

    if not obs.get("buyer_profile_stored"):
        if "diet" in remaining and "visit_availability" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your dietary preference and visit availability."}
        if "diet" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your dietary preference."}
        if "visit_availability" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your visit availability."}
        return {"action_type": "tool_call", "tool_name": "store_user_details", "tool_arguments": {}}

    post_ids = ["post_031", "post_052"] if scenario_id == "task_visit_multi" else ["post_031"]
    if "search_posts" not in tools:
        return {"action_type": "tool_call", "tool_name": "search_posts", "tool_arguments": {}}
    if "match_location_preference" not in tools:
        return {"action_type": "tool_call", "tool_name": "match_location_preference", "tool_arguments": {"post_ids": post_ids}}
    if "get_commute_time" not in tools:
        return {"action_type": "tool_call", "tool_name": "get_commute_time", "tool_arguments": {"post_ids": post_ids}}
    if "check_calendar_slots" not in tools:
        return {"action_type": "tool_call", "tool_name": "check_calendar_slots", "tool_arguments": {"post_ids": post_ids}}
    if "shortlist" not in tools:
        return {"action_type": "tool_call", "tool_name": "shortlist", "tool_arguments": {"post_ids": post_ids}}
    if "contact_poster" not in tools:
        return {"action_type": "tool_call", "tool_name": "contact_poster", "tool_arguments": {"post_id": post_ids[0], "time_text": "tomorrow 7pm"}}
    if "book_viewing" not in tools:
        return {"action_type": "tool_call", "tool_name": "book_viewing", "tool_arguments": {"post_id": post_ids[0], "time_text": "tomorrow 7pm"}}
    return None

def compact_observation(obs: dict[str, Any]) -> dict[str, Any]:
    return {
        "scenario_id": obs.get("scenario_id"),
        "phase": obs.get("phase"),
        "status": obs.get("status"),
        "last_user_message": obs.get("last_user_message"),
        "current_user_request": obs.get("current_user_request"),
        "available_tools": obs.get("available_tools", []),
        "remaining_required_fields": obs.get("remaining_required_fields", []),
        "prerequisites_satisfied": obs.get("prerequisites_satisfied", {}),
        "recent_tool_calls": obs.get("recent_tool_calls", []),
        "last_tool_result": obs.get("last_tool_result", {}),
        "violations": obs.get("violations", []),
        "booked_visits": obs.get("booked_visits", []),
        "feedback_summary": obs.get("feedback_summary", ""),
    }

def prompt_from_observation(obs: dict[str, Any]) -> str:
    return (
        "You are a broker policy for the Flatmate RL environment. "
        "Return exactly one JSON action and no extra text.\n\n"
        "Valid action shapes:\n"
        "{\"action_type\":\"assistant_message\",\"assistant_message\":\"...\"}\n"
        "{\"action_type\":\"tool_call\",\"tool_name\":\"...\",\"tool_arguments\":{...}}\n\n"
        f"Observation:\n{json.dumps(compact_observation(obs), ensure_ascii=False, sort_keys=True)}\n\n"
        "Action:\n"
    )

In [ ]:
@dataclass
class PromptCollectionConfig:
    episodes: int = 12
    max_steps: int = 14
    seed: int = 11

async def collect_prompt_rows(config: PromptCollectionConfig = PromptCollectionConfig()) -> list[dict[str, Any]]:
    rng = random.Random(config.seed)
    rows: list[dict[str, Any]] = []
    for episode_idx in range(config.episodes):
        scenario_id = rng.choice(SCENARIOS)
        prefix_actions: list[dict[str, Any]] = []
        async with FlatmateEndpoint() as env:
            obs = await env.reset(scenario_id, seed=config.seed + episode_idx)
            for step_idx in range(config.max_steps):
                action = heuristic_action(obs)
                if action is None or obs.get("done"):
                    break
                rows.append({
                    "prompt": prompt_from_observation(obs),
                    "scenario_id": scenario_id,
                    "seed": config.seed + episode_idx,
                    "prefix_actions_json": json.dumps(prefix_actions, ensure_ascii=False),
                    "reference_action_json": json.dumps(action, ensure_ascii=False, sort_keys=True),
                })
                obs = await env.step(action)
                prefix_actions.append(action)
                if obs.get("done"):
                    break
        print(f"episode={episode_idx:02d} scenario={scenario_id} total_rows={len(rows)}")
    return rows

rows = await collect_prompt_rows(PromptCollectionConfig(episodes=12, max_steps=14))
train_dataset = Dataset.from_list(rows)
train_dataset

## GRPO Rewards

`json_format_reward` is cheap and runs for every completion. `endpoint_transition_reward` is slower because it calls the live Space: it replays the prefix actions, sends the generated action, and returns a reward based on environment reward, validity, violations, and bookings.

In [ ]:
def completion_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list) and completion and isinstance(completion[0], dict):
        return str(completion[0].get("content", ""))
    return str(completion)

def parse_json_action(text: str) -> dict[str, Any]:
    text = completion_text(text).strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("completion does not contain a JSON object")
    action = json.loads(text[start : end + 1])
    if action.get("action_type") == "assistant_message":
        if not str(action.get("assistant_message", "")).strip():
            raise ValueError("assistant_message action needs assistant_message")
        return {
            "action_type": "assistant_message",
            "assistant_message": str(action["assistant_message"]),
        }
    if action.get("action_type") == "tool_call":
        if not str(action.get("tool_name", "")).strip():
            raise ValueError("tool_call action needs tool_name")
        args = action.get("tool_arguments", {})
        if not isinstance(args, dict):
            raise ValueError("tool_arguments must be an object")
        return {
            "action_type": "tool_call",
            "tool_name": str(action["tool_name"]),
            "tool_arguments": args,
        }
    raise ValueError("action_type must be assistant_message or tool_call")

def json_format_reward(completions, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        try:
            parse_json_action(completion_text(completion))
            rewards.append(0.25)
        except Exception:
            rewards.append(-1.0)
    return rewards

async def score_one_completion(
    completion: Any,
    scenario_id: str,
    seed: int,
    prefix_actions_json: str,
) -> float:
    try:
        action = parse_json_action(completion_text(completion))
        prefix_actions = json.loads(prefix_actions_json)
    except Exception:
        return -1.0

    try:
        async with FlatmateEndpoint() as env:
            obs = await env.reset(scenario_id, seed=int(seed))
            for prefix_action in prefix_actions:
                obs = await env.step(prefix_action)
                if obs.get("done"):
                    return -0.5

            before_violations = len(obs.get("violations", []))
            before_bookings = len(obs.get("booked_visits", []))
            obs = await env.step(action)

        reward = float(obs.get("reward") or obs.get("step_reward") or 0.0)
        reward += 0.15
        if len(obs.get("violations", [])) > before_violations:
            reward -= 0.75
        if len(obs.get("booked_visits", [])) > before_bookings:
            reward += 1.0
        if obs.get("done"):
            reward += 0.5
        return float(max(-2.0, min(2.0, reward)))
    except Exception as exc:
        print("endpoint reward error:", repr(exc))
        return -1.0

async def score_completion_batch(completions, scenario_id, seed, prefix_actions_json) -> list[float]:
    tasks = [
        score_one_completion(c, s, int(sd), p)
        for c, s, sd, p in zip(completions, scenario_id, seed, prefix_actions_json)
    ]
    return list(await asyncio.gather(*tasks))

def run_async_blocking(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    result: dict[str, Any] = {}
    def runner():
        try:
            result["value"] = asyncio.run(coro)
        except Exception as exc:
            result["error"] = exc

    thread = threading.Thread(target=runner, daemon=True)
    thread.start()
    thread.join()
    if "error" in result:
        raise result["error"]
    return result["value"]

def endpoint_transition_reward(completions, scenario_id, seed, prefix_actions_json, **kwargs) -> list[float]:
    return run_async_blocking(score_completion_batch(completions, scenario_id, seed, prefix_actions_json))

# Quick reward smoke test with known reference actions from the collected dataset.
sample = train_dataset.select(range(min(2, len(train_dataset))))
endpoint_transition_reward(
    completions=sample["reference_action_json"],
    scenario_id=sample["scenario_id"],
    seed=sample["seed"],
    prefix_actions_json=sample["prefix_actions_json"],
)

## Train with GRPO

The endpoint reward is network-bound, so keep `num_generations`, dataset size, and max steps small until the loop is stable. Increase them once you see valid JSON actions and non-negative endpoint rewards.

In [ ]:
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "flatmate-rl-grpo-policy"

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=1536,
    max_completion_length=160,
    max_steps=30,
    logging_steps=1,
    save_steps=15,
    save_total_limit=2,
    report_to="none",
)

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=[json_format_reward, endpoint_transition_reward],
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
)

train_result = trainer.train()
train_log_history = trainer.state.log_history
trainer.save_model(OUTPUT_DIR)
train_result

## Training Log

Plot logged GRPO reward and loss curves over optimizer steps.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

log_path = Path(OUTPUT_DIR) / "train_log_history.json"
log_path.parent.mkdir(parents=True, exist_ok=True)
log_path.write_text(json.dumps(train_log_history, indent=2))

def plot_training_log(log_history, title: str = "GRPO training log"):
    df = pd.DataFrame(log_history)
    if df.empty:
        print("No trainer log rows found yet.")
        return df
    metric_cols = [col for col in ["loss", "reward", "rewards/json_format_reward", "rewards/endpoint_transition_reward", "kl"] if col in df.columns]
    if not metric_cols:
        metric_cols = [col for col in df.columns if "reward" in col or col in {"loss", "kl"}]
    if not metric_cols or "step" not in df.columns:
        print("No plottable step metrics found. Available columns:", list(df.columns))
        return df
    axes = df.dropna(subset=["step"]).plot(
        x="step",
        y=metric_cols,
        marker="o",
        figsize=(9, 4),
        title=title,
    )
    axes.set_xlabel("optimizer step")
    axes.grid(True, alpha=0.3)
    plt.show()
    return df

train_log_df = plot_training_log(train_log_history)
train_log_df.tail()

## Evaluate One Episode

In [ ]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoPeftModelForCausalLM.from_pretrained(OUTPUT_DIR, device_map="auto")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.use_cache = False

def safe_model_action(obs: dict[str, Any]) -> dict[str, Any]:
    prompt = prompt_from_observation(obs)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536).to(model.device)
    model.generation_config.do_sample = False
    model.generation_config.temperature = None
    model.generation_config.top_p = None
    model.generation_config.top_k = None
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=96,
            do_sample=False,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    try:
        return parse_json_action(completion)
    except Exception as exc:
        fallback = heuristic_action(obs)
        if fallback is None:
            fallback = {"action_type": "assistant_message", "assistant_message": "Could you confirm the details needed for scheduling?"}
        print(f"bad generation, using fallback: {exc}")
        print("raw completion:", repr(completion[:300]))
        return fallback

def heuristic_policy(obs: dict[str, Any]) -> dict[str, Any]:
    action = heuristic_action(obs)
    if action is None:
        return {"action_type": "assistant_message", "assistant_message": "Could you confirm the details needed for scheduling?"}
    return action

async def evaluate_policy(policy_fn, label: str, scenarios=SCENARIOS, seeds=(123, 124), max_steps: int = 20, verbose: bool = False):
    rows = []
    for scenario_id in scenarios:
        for seed in seeds:
            async with FlatmateEndpoint() as env:
                obs = await env.reset(scenario_id, seed=seed)
                total_reward = 0.0
                steps = 0
                for step_idx in range(max_steps):
                    action = policy_fn(obs)
                    if verbose:
                        print(label, scenario_id, seed, step_idx, action)
                    obs = await env.step(action)
                    steps = step_idx + 1
                    total_reward += float(obs.get("reward") or obs.get("step_reward") or 0.0)
                    if obs.get("done"):
                        break
                rows.append({
                    "policy": label,
                    "scenario_id": scenario_id,
                    "seed": seed,
                    "total_reward": total_reward,
                    "done": bool(obs.get("done")),
                    "bookings": len(obs.get("booked_visits", [])),
                    "violations": len(obs.get("violations", [])),
                    "steps": steps,
                })
    return rows

heuristic_eval = await evaluate_policy(heuristic_policy, "heuristic")
grpo_eval = await evaluate_policy(safe_model_action, "grpo_trained")
eval_rows = heuristic_eval + grpo_eval
eval_df = pd.DataFrame(eval_rows)
eval_df

## Performance Comparison

Compare heuristic rollout behavior against the trained GRPO policy on the same scenarios and seeds.

In [ ]:
def plot_policy_comparison(eval_df, title: str = "GRPO policy comparison"):
    summary = (
        eval_df.groupby("policy", as_index=True)
        .agg(
            avg_reward=("total_reward", "mean"),
            completion_rate=("done", "mean"),
            avg_bookings=("bookings", "mean"),
            avg_violations=("violations", "mean"),
            avg_steps=("steps", "mean"),
        )
        .sort_index()
    )
    axes = summary[["avg_reward", "completion_rate", "avg_bookings", "avg_violations"]].plot(
        kind="bar",
        subplots=True,
        layout=(2, 2),
        figsize=(10, 7),
        legend=False,
        title=title,
    )
    for ax in axes.ravel():
        ax.grid(axis="y", alpha=0.3)
        ax.set_xlabel("")
    plt.tight_layout()
    plt.show()
    return summary

comparison_summary = plot_policy_comparison(eval_df)
comparison_summary

In [ ]:
# Optional Hub upload after training.
# from huggingface_hub import notebook_login
# notebook_login()
# trainer.push_to_hub("flatmate-rl-grpo-policy")